In [1]:
!git clone https://github.com/riyamaurya86/crowd-counting-partB.git

Cloning into 'crowd-counting-partB'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 223 (delta 36), reused 53 (delta 16), pack-reused 134 (from 1)
Receiving objects: 100% (223/223), 123.50 MiB | 8.90 MiB/s, done.
Resolving deltas: 100% (97/97), done.


In [2]:
%cd crowd-counting-partB

/kaggle/working/crowd-counting-partB


In [3]:
import os
import torch
from torch.utils.data import DataLoader

from src.datasets.shanghai_partb import ShanghaiPartBDataset
from src.models.csrnet_multiscale import CSRNet_MultiScale
from src.engine.trainer import train_one_epoch, validate, save_checkpoint
from src.losses.mse import get_mse_loss
from src.utils.seed import set_seed

In [4]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
dataset_path = "/kaggle/input/datasets/tthien/shanghaitech-with-people-density-map/ShanghaiTech/part_B"

train_dataset = ShanghaiPartBDataset(dataset_path, mode="train", crop_size=256)
test_dataset = ShanghaiPartBDataset(dataset_path, mode="test")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 400
Test size: 316


In [6]:
model = CSRNet_MultiScale(pretrained=True).to(device)

criterion = get_mse_loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 195MB/s] 


In [7]:
num_epochs = 50
best_mae = float("inf")

os.makedirs("checkpoints/csrnet_multiscale", exist_ok=True)

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_results = validate(model, test_loader, criterion, device)

    print(f"Train Loss: {train_loss:.6f}")
    print(f"Val MAE: {val_results['mae']:.2f}")
    print(f"Val RMSE: {val_results['rmse']:.2f}")
    print(f"Val PSNR: {val_results['psnr']:.2f}")
    print(f"Val SSIM: {val_results['ssim']:.4f}")

    # Save best model
    if val_results["mae"] < best_mae:
        best_mae = val_results["mae"]
        best_results = val_results.copy()
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_mae,
            "checkpoints/csrnet_multiscale/best_model.pth"
        )


Epoch [1/50]


Validating: 100%|██████████| 316/316 [00:39<00:00,  8.06it/s]


Train Loss: 0.000002
Val MAE: 70.13
Val RMSE: 95.73
Val PSNR: 20.89
Val SSIM: 0.2543

Epoch [2/50]


Validating: 100%|██████████| 316/316 [00:39<00:00,  8.10it/s]


Train Loss: 0.000001
Val MAE: 50.46
Val RMSE: 73.43
Val PSNR: 22.95
Val SSIM: 0.3137

Epoch [3/50]


...

Train Loss: 0.000000
Val MAE: 19.08
Val RMSE: 29.05
Val PSNR: 31.31
Val SSIM: 0.7256

Epoch [49/50]


Validating: 100%|██████████| 316/316 [00:38<00:00,  8.16it/s]


Train Loss: 0.000000
Val MAE: 58.50
Val RMSE: 69.93
Val PSNR: 30.58
Val SSIM: 0.7326

Epoch [50/50]


Validating: 100%|██████████| 316/316 [00:38<00:00,  8.13it/s]


Train Loss: 0.000000
Val MAE: 15.77
Val RMSE: 26.34
Val PSNR: 31.42
Val SSIM: 0.7319


In [9]:
import pandas as pd

results_df = pd.DataFrame([best_results])
results_df.to_csv("results/csrnet_multiscale_metrics.csv", index=False)

print("Saved results.")

Saved results.


In [10]:
import shutil

shutil.copy(
    "checkpoints/csrnet_multiscale/best_model.pth",
    "/kaggle/working/csrnet_multiscale_best_model.pth"
)

print("Checkpoint copied to working directory.")

Checkpoint copied to working directory.


In [11]:
from src.utils.visualization import visualize_predictions

fixed_indices = [165, 173, 33, 78, 93]

visualize_predictions(
    model,
    test_dataset,
    device,
    save_dir="results/qualitative_results/csrnet_multiscale",
    indices=fixed_indices
)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].


Saved visualizations to results/qualitative_results/csrnet_multiscale
